# MNIST Classification: Keras vs PyTorch Approaches
A comparison of three implementation strategies for MNIST digit classification

## Key Comparison Table

| Feature               | Keras                     | PyTorch Sequential        | PyTorch Custom            |
|-----------------------|---------------------------|---------------------------|---------------------------|
| **Code Verbosity**    | Low                       | Medium                    | High                      |
| **Training Interface**| `model.fit()`             | Custom Loops              | Custom Loops             |
| **Input Shape**       | First layer only          | Calculated automatically  | Explicit in forward pass  |
| **GPU Handling**      | Automatic                 | Manual                    | Manual                    |
| **Architecture Flex** | Limited                   | Moderate                  | Unlimited                 |
| **Debugging**         | Graph-based               | Eager Execution           | Full Python Debugging     |
| **Use Case**          | Production/Prototyping    | Simple Customization      | Research/Complex Models  |


## 1. Keras Implementation
- **Low verbosity**
- **fit and evaluate methods**: no user-defined loops
  - progress indicators:
    - epoch, batch within epoch
    - metrics(loss/accuracy)
      - train/evaluation datasets
- **Automatic input shape determination**
- **Automatic GPU handling** - No explicit device management required

# Keras works can multiple back-ends *without code changes*
- "backend-agnostic"

with a little care.

See
- https://keras.io/guides/migrating_to_keras_3/ for info on migrating from
Keras 2 (TensorFlow) to Keras 3.
- https://keras.io/guides/migrating_to_keras_3/#transitioning-to-backendagnostic-keras-3 on using backend-agnostic operations

To make this possible import from `keras` rather than `tensorflow.keras`.

For example:

```
  import keras
  from keras.layers import Input, Dense
```

rather than

```
  from tensorflow.keras.layers import Input, Dense
```

The environment variable KERAS_BACKEND controls the backend to use.  Setting:
- PyTorch: "torch"
- TensorFlow: "tensorflow"

In Colab: you need to restart the runtime if you switch the backend.


In [2]:
# prompt: set keras backend to pytorch

import os
os.environ["KERAS_BACKEND"] =  "torch" # "tensorflow"

import keras

print(f"Keras backend: {keras.backend.backend()}")


Keras backend: torch


In [3]:
# Keras Implementation
from keras.datasets import mnist
from keras import Sequential
from keras.layers import Input, Dense

# Data loading
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255
x_test = x_test.reshape(-1, 784).astype('float32') / 255

# Model definition
model = Sequential([
    Input(shape=(784,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Training
history = model.fit(x_train, y_train,
                    epochs=5,
                    batch_size=128,
                    validation_split=0.1)

# Evaluation
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f}')

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.8166 - loss: 0.6402 - val_accuracy: 0.9605 - val_loss: 0.1441
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9523 - loss: 0.1610 - val_accuracy: 0.9672 - val_loss: 0.1159
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9680 - loss: 0.1048 - val_accuracy: 0.9750 - val_loss: 0.0934
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9778 - loss: 0.0778 - val_accuracy: 0.9793 - val_loss: 0.0774
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9823 - loss: 0.0585 - val_accuracy: 0.9765 - val_loss: 0.0799

Test accuracy: 0.9729


## 2. PyTorch Sequential
- **Manual specification of shape:** must specify input dimension of layer as well as output dimension
- **User create training/evaluation loops**: compare to `fit` and `eval` methods of Keras
- **Manual device management** required for GPU acceleration



In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Convert data to PyTorch tensors
train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Model definition
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

# Training setup
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

# Training loop
for epoch in range(5):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
print(f'\nTest accuracy: {correct/len(test_dataset):.4f}')


Test accuracy: 0.9747


## 3. PyTorch Custom Model
- **Maximum flexibility** with custom forward pass
- **Forward method defines connectivity of layers**: compared to Sequential API of PyTorch or Keras

In [5]:
class CustomMNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.block1 = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU()
        )
        self.block2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

# Training (same as Sequential)
model = CustomMNISTModel().to(device)
optimizer = torch.optim.Adam(model.parameters())

for epoch in range(5):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
print(f'\nTest accuracy: {correct/len(test_dataset):.4f}')


Test accuracy: 0.9711


## Key Takeaways

- **Keras**: Best for rapid prototyping with minimal boilerplate
- **PyTorch Sequential**: Good balance between simplicity and customization
- **PyTorch Custom**: Essential for complex architectures and research
- **GPU Handling**: Automatic in Keras vs explicit in PyTorch
- **Production vs Research**: Keras for deployment, PyTorch for experimentation

# Create equivalent of Keras `.fit` and `.evaluate` in PyTorch
- greater code complexity
- these "loops" are present in every program !
  - repeated code unless we define functions in a library and re-use

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

# Data preparation
def get_data_loaders(batch_size=128, val_split=0.1):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1))  # Flatten 28x28 to 784
    ])

    train_dataset = datasets.MNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # Split validation set
    val_size = int(len(train_dataset) * val_split)
    train_size = len(train_dataset) - val_size
    train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

    test_dataset = datasets.MNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(val_dataset, batch_size=batch_size),
        DataLoader(test_dataset, batch_size=batch_size)
    )

# Model definition
def create_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

# Keras-style fit function
def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs=5, device='cpu'):
    model.to(device)
    history = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)

        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            # Forward pass
            outputs = model(X)
            loss = loss_fn(outputs, y)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Update metrics
            train_loss += loss.item() * X.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == y).sum().item()
            train_total += y.size(0)

            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{(train_correct/train_total):.2%}'
            })

        # Calculate epoch metrics
        train_loss /= train_total
        train_acc = train_correct / train_total
        history['loss'].append(train_loss)
        history['acc'].append(train_acc)

        # Validation phase
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch+1}/{epochs} - '
              f'loss: {train_loss:.4f} - acc: {train_acc:.2%} - '
              f'val_loss: {val_loss:.4f} - val_acc: {val_acc:.2%}')

    return history

# Keras-style evaluate function
def evaluate(model, data_loader, loss_fn, device='cpu'):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating', leave=False)
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)
            outputs = model(X)

            loss = loss_fn(outputs, y)
            total_loss += loss.item() * X.size(0)

            _, predicted = torch.max(outputs, 1)
            total_correct += (predicted == y).sum().item()
            total_samples += y.size(0)

            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{(total_correct/total_samples):.2%}'
            })

    return total_loss / total_samples, total_correct / total_samples

# Example usage
if __name__ == "__main__":
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Get data loaders
    train_loader, val_loader, test_loader = get_data_loaders()

    # Model setup
    model = create_model()
    optimizer = torch.optim.Adam(model.parameters())
    loss_fn = nn.CrossEntropyLoss()

    # Training
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        epochs=5,
        device=device
    )

    # Final evaluation
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    print(f'\nTest results - loss: {test_loss:.4f} - acc: {test_acc:.2%}')


Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 340kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.02MB/s]


Epoch 1/5:   0%|          | 0/422 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/47 [00:00<?, ?it/s]

Epoch 1/5 - loss: 0.3587 - acc: 90.31% - val_loss: 0.1818 - val_acc: 94.82%


...

Epoch 5/5:   0%|          | 0/422 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/47 [00:00<?, ?it/s]

Epoch 5/5 - loss: 0.0519 - acc: 98.42% - val_loss: 0.0944 - val_acc: 97.03%


Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]


Test results - loss: 0.0908 - acc: 97.19%
